Bronze to Silver Transformation

Objective
Transform raw USGS hourly JSON data into a structured Silver Delta table with consistent schema and deduplication.

Process Overview

Load latest hourly JSON data from Bronze layer (/Files/bronze_usgs_hourly) into a raw dataframe.
Flatten nested JSON structure by extracting fields from features, properties, and geometry.
Align transformed data to the predefined Silver schema to ensure consistency.
Merge new records into the earthquakes_silver table using event_id as the unique key.
Remove processed hourly files from Bronze layer to prevent duplicate ingestion.

In [3]:
from pyspark.sql.functions import explode, col, from_unixtime, to_timestamp
from delta.tables import DeltaTable

# Read the recently ingested hourly file in nb_usgs_ingest_bronze
file_path = "Files/bronze_usgs_hourly/*.json"

# Read latest hourly raw JSON
df_raw_latest = spark.read.option("multiline", "true").json(file_path)



# Read that exact hourly raw JSON file
df_raw_latest = spark.read.option("multiline", "true").json(file_path)


# Explode features
df_features_latest = df_raw_latest.select(explode("features").alias("feature"))

# Read Silver column order from existing earthquakes_silver table
silver_cols_bootstrap = spark.table("earthquakes_silver").columns
print("Silver bootstrap columns:", silver_cols_bootstrap)

# Flatten hourly data to match bootstrap schema
df_flat_latest = df_features_latest.select(
    col("feature.id").alias("event_id"),
    col("feature.type").alias("feature_type"),
    col("feature.properties.mag").alias("magnitude"),
    col("feature.properties.place").alias("place"),
    col("feature.properties.status").alias("status"),
    col("feature.properties.magType").alias("mag_type"),
    col("feature.properties.net").alias("net"),
    col("feature.properties.code").alias("code"),
    col("feature.properties.url").alias("url"),
    col("feature.geometry.type").alias("geometry_type"),
    col("feature.geometry.coordinates")[0].alias("longitude"),
    col("feature.geometry.coordinates")[1].alias("latitude"),
    col("feature.geometry.coordinates")[2].alias("depth_km"),
    to_timestamp(from_unixtime(col("feature.properties.time") / 1000)).alias("event_time_utc"),
    to_timestamp(from_unixtime(col("feature.properties.updated") / 1000)).alias("updated_time_utc")
)

# Match Silver column order
df_flat_latest = df_flat_latest.select(*silver_cols_bootstrap)

# Review incoming data
df_flat_latest.printSchema()
display(df_flat_latest)
print(f"Total incoming rows: {df_flat_latest.count()}")

# Count rows in earthquakes_silver before merge
silver_rows_before = spark.table("earthquakes_silver").count()
print(f"Total rows in earthquakes_silver before merge: {silver_rows_before}")

# Reference target Delta table
delta_target = DeltaTable.forName(spark, "earthquakes_silver")

# Merge hourly rows into Silver using event_id as natural key
(
    delta_target.alias("target").merge(df_flat_latest.alias("source"),"target.event_id = source.event_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

# Count rows in earthquakes_silver after merge
silver_rows_after = spark.table("earthquakes_silver").count()
print(f"Total rows in earthquakes_silver after merge: {silver_rows_after}")

# Optional: inspect final table
#display(spark.table("earthquakes_silver"))

# Delete processed hourly file/folder after successful merge
mssparkutils.fs.rm("Files/bronze_usgs_hourly/", True)
print("Processed hourly Bronze files deleted")






StatementMeta(, 0b609df6-d516-4e36-bec4-3e3bd649bd04, 5, Finished, Available, Finished, False)

Silver bootstrap columns: ['event_id', 'feature_type', 'magnitude', 'place', 'status', 'mag_type', 'net', 'code', 'url', 'geometry_type', 'longitude', 'latitude', 'depth_km', 'event_time_utc', 'updated_time_utc']
root
 |-- event_id: string (nullable = true)
 |-- feature_type: string (nullable = true)
 |-- magnitude: double (nullable = true)
 |-- place: string (nullable = true)
 |-- status: string (nullable = true)
 |-- mag_type: string (nullable = true)
 |-- net: string (nullable = true)
 |-- code: string (nullable = true)
 |-- url: string (nullable = true)
 |-- geometry_type: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- depth_km: double (nullable = true)
 |-- event_time_utc: timestamp (nullable = true)
 |-- updated_time_utc: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, dbf1957c-d4a0-4b8d-b0ed-06d5ac179dbb)

Total incoming rows: 12
Total rows in earthquakes_silver before merge: 10723
Total rows in earthquakes_silver after merge: 10723
Processed hourly Bronze files deleted
